# Dependencias e Instalación

## Dependencias

In [ ]:
!uv pip install torch torchaudio torchcodec --torch-backend=auto

## Instalación

In [ ]:
!uv pip install -e RUTA_MODEL

In [ ]:
!uv pip install -e RUTA_MODEL/[languages]

## Para evitar incompatibilidades

In [ ]:
!uv pip install transformers==5.0.0

## Para configurar el dataset

In [ ]:
!uv pip install librosa soundfile

# Preparar el dataset

Hay que preparar el dataset con archivos `.wav` y las frases en un mismo `.csv`.

Estructura del dataset:
```
dataset
  ├── metadata.csv
  └── wavs
        ├── example1.wav
        └── example2.wav
```

Información en `metadata.csv`
```
example1|Transcription1
example2|Transcription2
```

> Nota: *Sobre los archivos de audio, de preferencia, para evitar pérdidas de calidad, las grabaciones deberían realizarse en `.wav` con una frecuencia de muestreo de `22050Hz`. Debido a que es lo utilizado para el entrenamiento.*

A continuación se muestra un código en python para crear el archivo `.csv` y remuestrear los `.wav` si es necesario.

In [ ]:
import os
from pathlib import Path
import librosa
import soundfile as sf

def main(input_dir, output_dir, encoding_txt, extension_audio, target_sr=22050):
    input_dir = Path(input_dir)
    output_dir = Path(output_dir)

    if not input_dir.is_dir():
        raise ValueError(f"El directorio {input_dir} no existe.")

    # Crear directorio de salida y subcarpeta wavs (si no existen)
    output_dir.mkdir(parents=True, exist_ok=True)
    wavs_dir = output_dir / "wavs"
    wavs_dir.mkdir(exist_ok=True)

    wav_files = list(input_dir.glob(f"*.{extension_audio}"))
    if not wav_files:
        print(f"No se encontraron archivos con extensión .wav en {input_dir}")
        return

    with open(output_dir / "metadata.csv", "w", encoding="utf-8") as out_f:
        for wav_path in wav_files:
            txt_path = wav_path.with_suffix(".txt")
            if not txt_path.exists():
                print(f"No se encuentra transcripción para {wav_path.name}. Se omite.")
                continue

            # Leer transcripción
            text = txt_path.read_text(encoding=encoding_txt).strip()

            # Ruta de salida para el audio (mismo nombre)
            target_wav_path = wavs_dir / wav_path.name

            try:
                # Cargar audio (mono, frecuencia original)
                audio, sr = librosa.load(wav_path, sr=None, mono=True)

                # Resamplear si la frecuencia no es la deseada
                if sr != target_sr:
                    audio = librosa.resample(audio, orig_sr=sr, target_sr=target_sr)

                # Guardar como WAV (16-bit PCM)
                sf.write(target_wav_path, audio, target_sr, subtype='PCM_16')

                # Escribir metadatos (formato: nombre_sin_ext|texto|texto)
                out_f.write(f"{wav_path.stem}|{text}|{text}\n")

            except Exception as e:
                print(f"Error procesando {wav_path.name}: {e}")
                continue

if __name__ == "__main__":
    input_dir = "RUTA_DATASET"
    output_dir = "/kaggle/working/dataset"
    extension_audio = "wav"
    encoding_txt = "windows-1252"

    main(input_dir, output_dir, encoding_txt, extension_audio)
    print("Dataset organizado correctamente")

# Entrenamiento de XTTSv2

## Crear archivo para entrenar

In [ ]:
%%writefile finetunning.py
import os

from trainer import Trainer, TrainerArgs

from TTS.config.shared_configs import BaseDatasetConfig
from TTS.tts.configs.xtts_config import XttsAudioConfig
from TTS.tts.datasets import load_tts_samples
from TTS.tts.layers.xtts.trainer.gpt_trainer import GPTArgs, GPTTrainer, GPTTrainerConfig
from TTS.utils.manage import ModelManager

# Logging parameters
RUN_NAME = "GPT_XTTS_v2.0_LJSpeech_FT"
PROJECT_NAME = "XTTS_trainer"
DASHBOARD_LOGGER = "tensorboard"
LOGGER_URI = None

# Set here the path that the checkpoints will be saved. Default: ./run/training/
OUT_PATH = os.path.join(os.path.dirname(os.path.abspath(__file__)), "run", "training")

# Training Parameters
OPTIMIZER_WD_ONLY_ON_WEIGHTS = True  # for multi-gpu training please make it False
START_WITH_EVAL = True  # if True it will star with evaluation
BATCH_SIZE = 3  # set here the batch size
GRAD_ACUMM_STEPS = 84  # set here the grad accumulation steps
# Note: we recommend that BATCH_SIZE * GRAD_ACUMM_STEPS need to be at least 252 for more efficient training. You can increase/decrease BATCH_SIZE but then set GRAD_ACUMM_STEPS accordingly.

# Define here the dataset that you want to use for the fine-tuning on.
config_dataset = BaseDatasetConfig(
    formatter="ljspeech",
    dataset_name="ljspeech",
    path="/kaggle/working/dataset",
    meta_file_train="/kaggle/working/dataset/metadata.csv",
    language="es",
)

# Add here the configs of the datasets
DATASETS_CONFIG_LIST = [config_dataset]

# Define the path where XTTS v2.0.1 files will be downloaded
CHECKPOINTS_OUT_PATH = os.path.join(OUT_PATH, "XTTS_v2.0_original_model_files/")
os.makedirs(CHECKPOINTS_OUT_PATH, exist_ok=True)

# DVAE files
DVAE_CHECKPOINT_LINK = "https://huggingface.co/coqui/XTTS-v2/resolve/main/dvae.pth"
MEL_NORM_LINK = "https://huggingface.co/coqui/XTTS-v2/resolve/main/mel_stats.pth"

# Set the path to the downloaded files
DVAE_CHECKPOINT = os.path.join(CHECKPOINTS_OUT_PATH, os.path.basename(DVAE_CHECKPOINT_LINK))
MEL_NORM_FILE = os.path.join(CHECKPOINTS_OUT_PATH, os.path.basename(MEL_NORM_LINK))

# download DVAE files if needed
if not os.path.isfile(DVAE_CHECKPOINT) or not os.path.isfile(MEL_NORM_FILE):
    print(" > Downloading DVAE files!")
    ModelManager._download_model_files([MEL_NORM_LINK, DVAE_CHECKPOINT_LINK], CHECKPOINTS_OUT_PATH, progress_bar=True)


# Download XTTS v2.0 checkpoint if needed
TOKENIZER_FILE_LINK = "https://huggingface.co/coqui/XTTS-v2/resolve/main/vocab.json"
XTTS_CHECKPOINT_LINK = "https://huggingface.co/coqui/XTTS-v2/resolve/main/model.pth"

# XTTS transfer learning parameters: You we need to provide the paths of XTTS model checkpoint that you want to do the fine tuning.
TOKENIZER_FILE = os.path.join(CHECKPOINTS_OUT_PATH, os.path.basename(TOKENIZER_FILE_LINK))  # vocab.json file
XTTS_CHECKPOINT = os.path.join(CHECKPOINTS_OUT_PATH, os.path.basename(XTTS_CHECKPOINT_LINK))  # model.pth file

# download XTTS v2.0 files if needed
if not os.path.isfile(TOKENIZER_FILE) or not os.path.isfile(XTTS_CHECKPOINT):
    print(" > Downloading XTTS v2.0 files!")
    ModelManager._download_model_files(
        [TOKENIZER_FILE_LINK, XTTS_CHECKPOINT_LINK], CHECKPOINTS_OUT_PATH, progress_bar=True
    )


# Training sentences generations
SPEAKER_REFERENCE = [
    "RUTA_REFERENCIA" # speaker reference to be used in training test sentences
]
LANGUAGE = config_dataset.language


def main():
    # init args and config
    model_args = GPTArgs(
        max_conditioning_length=132300,  # 6 secs
        min_conditioning_length=66150,  # 3 secs
        debug_loading_failures=False,
        max_wav_length=441000,  # ~20 seconds
        max_text_length=300,
        mel_norm_file=MEL_NORM_FILE,
        dvae_checkpoint=DVAE_CHECKPOINT,
        xtts_checkpoint=XTTS_CHECKPOINT,  # checkpoint path of the model that you want to fine-tune
        tokenizer_file=TOKENIZER_FILE,
        gpt_num_audio_tokens=1026,
        gpt_start_audio_token=1024,
        gpt_stop_audio_token=1025,
        gpt_use_masking_gt_prompt_approach=True,
        gpt_use_perceiver_resampler=True,
    )
    # define audio config
    audio_config = XttsAudioConfig(sample_rate=22050, dvae_sample_rate=22050, output_sample_rate=24000)
    # training parameters config
    config = GPTTrainerConfig(
        output_path=OUT_PATH,
        model_args=model_args,
        run_name=RUN_NAME,
        project_name=PROJECT_NAME,
        run_description="""
            GPT XTTS training
            """,
        dashboard_logger=DASHBOARD_LOGGER,
        logger_uri=LOGGER_URI,
        audio=audio_config,
        batch_size=BATCH_SIZE,
        batch_group_size=48,
        eval_batch_size=BATCH_SIZE,
        num_loader_workers=4,
        eval_split_max_size=256,
        print_step=50,
        plot_step=500,
        log_model_step=5000,
        save_step=25000,
        save_n_checkpoints=1,
        save_checkpoints=True,
        # target_loss="loss",
        print_eval=False,
        # Optimizer values like tortoise, pytorch implementation with modifications to not apply WD to non-weight parameters.
        optimizer="AdamW",
        optimizer_wd_only_on_weights=OPTIMIZER_WD_ONLY_ON_WEIGHTS,
        optimizer_params={"betas": [0.9, 0.96], "eps": 1e-8, "weight_decay": 1e-2},
        lr=5e-06,  # learning rate
        lr_scheduler="MultiStepLR",
        # it was adjusted accordly for the new step scheme
        lr_scheduler_params={"milestones": [50000 * 18, 150000 * 18, 300000 * 18], "gamma": 0.5, "last_epoch": -1},
        test_sentences=[
            {
                "text": "TRANSCRIPCION",
                "speaker_wav": SPEAKER_REFERENCE,
                "language": LANGUAGE,
            },
        ],
    )

    # init the model from config
    model = GPTTrainer.init_from_config(config)

    # load training samples
    train_samples, eval_samples = load_tts_samples(
        DATASETS_CONFIG_LIST,
        eval_split=True,
        eval_split_max_size=config.eval_split_max_size,
        eval_split_size=config.eval_split_size,
    )

    # init the trainer and 🚀
    trainer = Trainer(
        TrainerArgs(
            restore_path=None,  # xtts checkpoint is restored via xtts_checkpoint key so no need of restore it using Trainer restore_path parameter
            skip_train_epoch=False,
            start_with_eval=START_WITH_EVAL,
            grad_accum_steps=GRAD_ACUMM_STEPS,
        ),
        config,
        output_path=OUT_PATH,
        model=model,
        train_samples=train_samples,
        eval_samples=eval_samples,
    )
    trainer.fit()


if __name__ == "__main__":
    main()

## Ejecutar entrenamiento

In [ ]:
!CUDA_VISIBLE_DEVICES="0" python finetunning.py

# Inferencia

In [ ]:
import sys
sys.path.append("RUTA_COQUITTS")

import os
import torch
import torchaudio
from TTS.tts.configs.xtts_config import XttsConfig
from TTS.tts.models.xtts import Xtts

# Add here the xtts_config path
CONFIG_PATH = "RUTA config.json"
# Add here the vocab file that you have used to train the model
TOKENIZER_PATH = "RUTA vocab.json"
# Add here the checkpoint that you want to do inference with
XTTS_CHECKPOINT = "RUTA best_model.pth"
# Add here the speaker reference
SPEAKER_REFERENCE = ["RUTA AUDIO DE REFERENCIA"]

print("Loading model...")
config = XttsConfig()
config.load_json(CONFIG_PATH)
model = Xtts.init_from_config(config)
model.load_checkpoint(config, checkpoint_path=XTTS_CHECKPOINT, vocab_path=TOKENIZER_PATH, use_deepspeed=False)
model.cuda() #ACTIVAR PARA GPU

print("Computing speaker latents...")
gpt_cond_latent, speaker_embedding = model.get_conditioning_latents(audio_path=SPEAKER_REFERENCE)

In [ ]:
text = """¡Hola!, esta es una voz artificial con el modelo de texto a voz entrenado para el t c u 748."""
# output wav path
OUTPUT_WAV_PATH = "/kaggle/working/out.wav"

print("Inference...")
out = model.inference(
    text,
    "es",
    gpt_cond_latent,
    speaker_embedding,
    temperature=0.95,
    top_k=50,
    top_p=0.8,
    repetition_penalty=1.0,
)
torchaudio.save(OUTPUT_WAV_PATH, torch.tensor(out["wav"]).unsqueeze(0), 24000)

In [ ]:
import IPython

IPython.display.Audio("/kaggle/working/out.wav")

# Streaming (Aún por probar)

Las siguientes líneas fueron generadas en su mayoría con ayuda de un LLM (DeepSeek), el código es funcional para streaming en tiempo real, con algunos errores por lo que posiblemente requiere mejoras importantes para mejorar las funciones implementadas.

In [ ]:
!uv pip install fastapi uvicorn websockets pyngrok

In [ ]:
import sys
sys.path.append("RUTA_COQUITTS")

import asyncio
import time
import torch
import torchaudio
from fastapi import FastAPI, WebSocket, WebSocketDisconnect
from fastapi.responses import HTMLResponse
import uvicorn

from TTS.tts.configs.xtts_config import XttsConfig
from TTS.tts.models.xtts import Xtts

# ------------------------------------------------------------
# CARGA DEL MODELO (se ejecuta una vez)
# ------------------------------------------------------------
print("Cargando modelo XTTS...")
CONFIG_PATH = "RUTA config.json"
TOKENIZER_PATH = "RUTA vocab.json"
XTTS_CHECKPOINT = "RUTA best_model.pth"
SPEAKER_REFERENCE = ["RUTA AUDIO DE REFERENCIA"]

config = XttsConfig()
config.load_json(CONFIG_PATH)
model = Xtts.init_from_config(config)
model.load_checkpoint(config, checkpoint_path=XTTS_CHECKPOINT, vocab_path=TOKENIZER_PATH, use_deepspeed=False)
model.cuda()

print("Calculando latentes del hablante...")
gpt_cond_latent, speaker_embedding = model.get_conditioning_latents(audio_path=SPEAKER_REFERENCE)
print("Modelo listo.\n")

In [ ]:
# ------------------------------------------------------------
# APLICACIÓN FASTAPI
# ------------------------------------------------------------
app = FastAPI()

@app.get("/")
async def get():
    html_content = """
    <!DOCTYPE html>
    <html>
    <head>
        <meta charset="UTF-8">
        <title>XTTS Live Streaming (Buffer optimizado)</title>
        <style>
            body { font-family: Arial, sans-serif; margin: 20px; }
            textarea { width: 100%; font-size: 16px; padding: 10px; }
            button { margin: 5px; padding: 8px 16px; font-size: 14px; }
            #status { margin-top: 10px; color: #555; }
            .auto-active { background-color: #4CAF50; color: white; }
        </style>
    </head>
    <body>
        <h2>🎙️ XTTS - Textos largos con buffer de reproducción (máx. 150 caracteres/fragmento)</h2>
        <textarea id="texto" rows="6" placeholder="Escribe aquí... (se dividirá en fragmentos de hasta 150 caracteres)"></textarea><br>
        
        <button id="toggleAuto">🔛 Activar modo automático (1s delay)</button>
        <button id="iniciarManual">▶️ Generar ahora</button>
        <button id="detener">⏹️ Detener todo</button>
        
        <p id="status">Estado: Listo. Escribe o usa los botones.</p>
        <p id="fragmentInfo" style="color: #666; font-size: 14px;"></p>
        
        <script>
            // ---------- CONFIGURACIÓN ----------
            const SAMPLE_RATE = 24000;
            const DEBOUNCE_DELAY = 1000;            // 1 segundo sin escribir
            const MAX_CHARS_PER_CHUNK = 150;        // Caracteres por fragmento de texto
            const MIN_BUFFER_CHUNKS = 3;            // Número mínimo de chunks antes de empezar a reproducir
            const BUFFER_TIMEOUT_MS = 500;          // Tiempo máximo de espera antes de reproducir aunque no se alcance MIN_BUFFER_CHUNKS
            
            // ---------- ELEMENTOS DOM ----------
            const textarea = document.getElementById('texto');
            const toggleAutoBtn = document.getElementById('toggleAuto');
            const iniciarManualBtn = document.getElementById('iniciarManual');
            const detenerBtn = document.getElementById('detener');
            const statusEl = document.getElementById('status');
            const fragmentInfo = document.getElementById('fragmentInfo');
            
            // ---------- ESTADO GLOBAL ----------
            let autoMode = false;
            let audioCtx = null;
            let debounceTimer = null;
            let isGenerating = false;
            let currentWS = null;
            
            // Fragmentos de texto
            let textFragments = [];
            let currentFragmentIndex = 0;
            
            // Sistema de buffer y reproducción
            let pendingBuffers = [];                // Cola de AudioBuffers listos para reproducir
            let playbackTimer = null;               // Timer del bucle de reproducción
            let nextPlayTime = 0;                   // Tiempo (audioCtx.currentTime) para el próximo buffer
            let playbackActive = false;             // Indica si el bucle está corriendo
            let activeSourceNodes = [];             // Para poder detenerlos manualmente
            
            // Control de inicio de reproducción
            let bufferAccumulatedTime = 0;          // Duración total acumulada en pendingBuffers
            let firstChunkArrived = false;          // Bandera para medir timeout desde el primer chunk
            
            // ---------- FUNCIONES AUXILIARES ----------
            function updateStatus(msg) {
                statusEl.innerText = 'Estado: ' + msg;
                console.log('Status:', msg);
            }
            
            function updateFragmentInfo() {
                if (textFragments.length > 0) {
                    fragmentInfo.innerText = `Texto dividido en ${textFragments.length} fragmento(s) de máx. ${MAX_CHARS_PER_CHUNK} caracteres.`;
                } else {
                    fragmentInfo.innerText = '';
                }
            }
            
            async function initAudio() {
                if (!audioCtx) {
                    audioCtx = new (window.AudioContext || window.webkitAudioContext)();
                }
                if (audioCtx.state === 'suspended') {
                    await audioCtx.resume();
                }
                // Reiniciar tiempo de referencia
                nextPlayTime = audioCtx.currentTime;
            }
            
            // Detener toda reproducción y limpiar estado
            function stopAllPlayback() {
                // Cancelar debounce
                if (debounceTimer) {
                    clearTimeout(debounceTimer);
                    debounceTimer = null;
                }
                // Detener bucle de reproducción
                if (playbackTimer) {
                    clearTimeout(playbackTimer);
                    playbackTimer = null;
                }
                playbackActive = false;
                
                // Cerrar WebSocket
                if (currentWS) {
                    currentWS.close();
                    currentWS = null;
                }
                
                // Detener todos los nodos de audio activos
                if (audioCtx) {
                    activeSourceNodes.forEach(node => {
                        try { node.stop(); } catch(e) {}
                    });
                    activeSourceNodes = [];
                }
                
                // Limpiar buffers pendientes
                pendingBuffers = [];
                bufferAccumulatedTime = 0;
                firstChunkArrived = false;
                
                isGenerating = false;
                currentFragmentIndex = 0;
                textFragments = [];
                updateFragmentInfo();
            }
            
            // Dividir texto en fragmentos por caracteres, respetando palabras
            function splitTextIntoChunks(text, maxChars) {
                const chunks = [];
                let remaining = text.trim();
                
                while (remaining.length > 0) {
                    if (remaining.length <= maxChars) {
                        chunks.push(remaining);
                        break;
                    }
                    
                    let cutIndex = remaining.lastIndexOf(' ', maxChars);
                    if (cutIndex === -1 || cutIndex === 0) {
                        cutIndex = maxChars;
                    }
                    
                    chunks.push(remaining.substring(0, cutIndex).trim());
                    remaining = remaining.substring(cutIndex).trim();
                }
                
                return chunks.filter(chunk => chunk.length > 0);
            }
            
            // Bucle de reproducción: se ejecuta periódicamente para encolar buffers cuando sea el momento
            function playbackLoop() {
                if (!playbackActive) return;
                
                const now = audioCtx.currentTime;
                
                // Si no hay buffers pendientes, esperar un poco y reintentar
                if (pendingBuffers.length === 0) {
                    playbackTimer = setTimeout(playbackLoop, 20);
                    return;
                }
                
                // Si aún no es momento de reproducir el siguiente buffer, esperar
                if (nextPlayTime > now + 0.01) {
                    playbackTimer = setTimeout(playbackLoop, Math.max(5, (nextPlayTime - now) * 1000 - 5));
                    return;
                }
                
                // Tomar el siguiente buffer de la cola
                const audioBuffer = pendingBuffers.shift();
                
                // Crear nodo fuente
                const sourceNode = audioCtx.createBufferSource();
                sourceNode.buffer = audioBuffer;
                sourceNode.connect(audioCtx.destination);
                
                // Ajustar tiempo de inicio (si nextPlayTime está en el pasado, iniciar ahora)
                const startTime = Math.max(now, nextPlayTime);
                sourceNode.start(startTime);
                
                // Actualizar nextPlayTime para el siguiente buffer
                nextPlayTime = startTime + audioBuffer.duration;
                
                // Guardar referencia para poder detenerlo
                activeSourceNodes.push(sourceNode);
                sourceNode.onended = () => {
                    const idx = activeSourceNodes.indexOf(sourceNode);
                    if (idx > -1) activeSourceNodes.splice(idx, 1);
                };
                
                // Programar siguiente iteración
                playbackTimer = setTimeout(playbackLoop, 10);
            }
            
            // Iniciar el bucle de reproducción cuando tengamos suficientes buffers
            function tryStartPlayback() {
                if (playbackActive) return; // Ya está corriendo
                
                // Criterios para comenzar:
                // - Al menos MIN_BUFFER_CHUNKS chunks, O
                // - Ha pasado BUFFER_TIMEOUT_MS desde que llegó el primer chunk
                const enoughChunks = pendingBuffers.length >= MIN_BUFFER_CHUNKS;
                const timeoutExceeded = firstChunkArrived && (performance.now() - firstChunkArrived > BUFFER_TIMEOUT_MS);
                
                if (enoughChunks || timeoutExceeded) {
                    playbackActive = true;
                    playbackLoop();
                    updateStatus('Reproducción iniciada (buffer listo)');
                }
            }
            
            // Añadir un chunk de audio al buffer y evaluar inicio de reproducción
            function enqueueAudioChunk(audioData) {
                // Decodificar los datos Float32Array a AudioBuffer
                const chunkData = new Float32Array(audioData);
                const audioBuffer = audioCtx.createBuffer(1, chunkData.length, SAMPLE_RATE);
                audioBuffer.copyToChannel(chunkData, 0);
                
                pendingBuffers.push(audioBuffer);
                bufferAccumulatedTime += audioBuffer.duration;
                
                if (!firstChunkArrived) {
                    firstChunkArrived = performance.now();
                }
                
                tryStartPlayback();
            }
            
            // Procesa el siguiente fragmento de texto
            async function processNextFragment() {
                if (!isGenerating) return;
                
                if (currentFragmentIndex >= textFragments.length) {
                    updateStatus('Todos los fragmentos generados. Esperando fin de reproducción...');
                    // No cerrar WebSocket aún, esperar a que termine la reproducción
                    // Cuando pendingBuffers esté vacío, se puede cerrar.
                    const checkDone = setInterval(() => {
                        if (!playbackActive || pendingBuffers.length === 0) {
                            clearInterval(checkDone);
                            if (currentWS) {
                                currentWS.close();
                                currentWS = null;
                            }
                            isGenerating = false;
                            updateStatus('Reproducción completada.');
                        }
                    }, 100);
                    return;
                }
                
                const fragmentText = textFragments[currentFragmentIndex];
                updateStatus(`Solicitando fragmento ${currentFragmentIndex+1}/${textFragments.length}: "${fragmentText.substring(0, 40)}..."`);
                
                // Si no hay WebSocket abierto, crear uno nuevo
                if (!currentWS || currentWS.readyState !== WebSocket.OPEN) {
                    const wsProtocol = location.protocol === 'https:' ? 'wss:' : 'ws:';
                    currentWS = new WebSocket(`${wsProtocol}//${location.host}/ws`);
                    currentWS.binaryType = 'arraybuffer';
                    
                    currentWS.onopen = () => {
                        currentWS.send(fragmentText);
                    };
                    
                    currentWS.onmessage = (event) => {
                        if (typeof event.data === 'string') {
                            if (event.data === 'END_OF_STREAM') {
                                // Fragmento de texto completado, pasar al siguiente
                                currentFragmentIndex++;
                                processNextFragment();
                            }
                            return;
                        }
                        // Datos binarios: chunk de audio
                        enqueueAudioChunk(event.data);
                    };
                    
                    currentWS.onerror = (err) => {
                        updateStatus(`Error en WebSocket para fragmento ${currentFragmentIndex+1}.`);
                        console.error(err);
                        stopAllPlayback();
                    };
                    
                    currentWS.onclose = () => {
                        if (isGenerating && currentFragmentIndex < textFragments.length) {
                            updateStatus('Conexión cerrada inesperadamente. Generación detenida.');
                            stopAllPlayback();
                        }
                    };
                } else {
                    // WebSocket ya abierto, enviar el texto
                    currentWS.send(fragmentText);
                }
            }
            
            // Inicia el proceso de generación para un texto completo
            async function startTextGeneration(fullText) {
                if (!fullText.trim()) {
                    updateStatus('Texto vacío, no se genera nada.');
                    return;
                }
                
                stopAllPlayback();
                
                await initAudio();
                
                textFragments = splitTextIntoChunks(fullText, MAX_CHARS_PER_CHUNK);
                updateFragmentInfo();
                
                if (textFragments.length === 0) return;
                
                isGenerating = true;
                currentFragmentIndex = 0;
                playbackActive = false;  // Se activará cuando haya suficientes buffers
                pendingBuffers = [];
                bufferAccumulatedTime = 0;
                firstChunkArrived = false;
                
                processNextFragment();
            }
            
            // Debounce para modo automático
            function handleTextInput() {
                if (!autoMode) return;
                
                if (debounceTimer) clearTimeout(debounceTimer);
                
                debounceTimer = setTimeout(() => {
                    if (autoMode) {
                        const texto = textarea.value;
                        if (texto.trim()) {
                            updateStatus('Texto estabilizado, iniciando generación...');
                            startTextGeneration(texto);
                        }
                    }
                    debounceTimer = null;
                }, DEBOUNCE_DELAY);
                
                updateStatus(`Auto modo activo. Esperando ${DEBOUNCE_DELAY/1000}s sin cambios...`);
            }
            
            // ---------- EVENT LISTENERS ----------
            textarea.addEventListener('input', handleTextInput);
            
            toggleAutoBtn.addEventListener('click', () => {
                autoMode = !autoMode;
                if (autoMode) {
                    toggleAutoBtn.textContent = '🔴 Desactivar modo automático';
                    toggleAutoBtn.classList.add('auto-active');
                    updateStatus('Modo automático ACTIVADO. Escribe para generar audio.');
                    if (textarea.value.trim()) {
                        handleTextInput();
                    }
                } else {
                    toggleAutoBtn.textContent = '🔛 Activar modo automático (1s delay)';
                    toggleAutoBtn.classList.remove('auto-active');
                    updateStatus('Modo automático DESACTIVADO.');
                    if (debounceTimer) {
                        clearTimeout(debounceTimer);
                        debounceTimer = null;
                    }
                }
            });
            
            iniciarManualBtn.addEventListener('click', () => {
                if (autoMode && debounceTimer) {
                    clearTimeout(debounceTimer);
                    debounceTimer = null;
                }
                const texto = textarea.value;
                if (!texto.trim()) {
                    updateStatus('Escribe algo para generar.');
                    return;
                }
                updateStatus('Generación manual iniciada...');
                startTextGeneration(texto);
            });
            
            detenerBtn.addEventListener('click', () => {
                stopAllPlayback();
                updateStatus('Reproducción detenida por el usuario.');
            });
            
            window.addEventListener('beforeunload', () => {
                if (currentWS) currentWS.close();
            });
            
            updateStatus('Listo. Activa el modo automático o usa "Generar ahora".');
        </script>
    </body>
    </html>
    """
    return HTMLResponse(content=html_content)

@app.websocket("/ws")
async def websocket_endpoint(websocket: WebSocket):
    await websocket.accept()
    print("Cliente conectado")
    try:
        while True:
            texto = await websocket.receive_text()
            print(f"Texto recibido: {texto[:80]}...")
            
            t0 = time.time()
            chunks = model.inference_stream(
                texto,
                "es",
                gpt_cond_latent,
                speaker_embedding
            )
            
            for i, chunk in enumerate(chunks):
                if i == 0:
                    print(f"Tiempo hasta primer chunk: {time.time() - t0:.3f}s")
                print(f"Enviando chunk {i} de {chunk.shape[-1]} muestras")
                
                chunk_np = chunk.squeeze().cpu().numpy().astype('float32')
                chunk_bytes = chunk_np.tobytes()
                await websocket.send_bytes(chunk_bytes)
                await asyncio.sleep(0.001)
            
            await websocket.send_text("END_OF_STREAM")
            print("Fragmento completado. Esperando siguiente texto...")
            
    except WebSocketDisconnect:
        print("Cliente desconectado.")
    except Exception as e:
        print(f"Error en WebSocket: {e}")
    finally:
        print("Conexión WebSocket finalizada.")

print("✅ Aplicación FastAPI definida correctamente.")

In [ ]:
# Celda 3 (reinicio forzado del servidor)
import nest_asyncio
import uvicorn
import threading
import time
import os
import signal
from pyngrok import ngrok

nest_asyncio.apply()

# (Opcional) Autenticación de ngrok (recomendada para evitar límites)
ngrok.set_auth_token("USAR API DE NGROK")

# ------------------------------------------------------------
# 1. Cerrar túneles ngrok anteriores
# ------------------------------------------------------------
try:
    ngrok.kill()
    print("🧹 Túneles ngrok anteriores cerrados.")
except Exception as e:
    print("No había túneles activos o error al cerrar:", e)

# ------------------------------------------------------------
# 2. Liberar el puerto 8000 (en Kaggle funciona con fuser)
# ------------------------------------------------------------
print("🔁 Liberando puerto 8000...")
os.system("fuser -k 8000/tcp > /dev/null 2>&1")

# ------------------------------------------------------------
# 3. Lanzar el servidor con la versión actualizada de 'app'
# ------------------------------------------------------------
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(3)

# ------------------------------------------------------------
# 4. Crear nuevo túnel público
# ------------------------------------------------------------
public_url = ngrok.connect(8000)
print(f"🌐 NUEVA URL pública: {public_url}")